# 🧹 Phase 1: Data Cleaning & Pre-processing

This notebook implements a high-performance, **parallelized cleaning pipeline** designed to handle large anime datasets efficiently.

### 🚀 Key Optimization Strategies
1.  **Parallel Processing**: CPU-intensive tasks (cleaning, remapping, optimization) are distributed across all available cores using `joblib`.
2.  **Memory Management**: 
    -   **`Clean` Phase**: Standardizes strings and deduplicates. This temporarily *increases* memory usage as efficient partial-loaded structures are converted to native Python objects.
    -   **`Optimization` Phase**: Aggressively downcasts numbers (e.g., `float64` -> `Int32`) and converts low-cardinality strings to `Categoricals`, resulting in massive memory savings (up to 85%).
3.  **Referential Integrity**: A global `ID MAP` is built progressively to ensure all datasets reference valid canonical IDs (merging duplicates).

### 📋 Workflow
1.  **Load** raw CSVs (using PyArrow where possible for speed).
2.  **Clean** (Standardize, Deduplicate, Fix Dates).
3.  **Remap Foreign Keys** (ensure relationships point to valid IDs).
4.  **Optimize Types**.
5.  **Save** to compressed Parquet format.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import gc
from pathlib import Path
import pprint
from joblib import Parallel, delayed

# 🧪 Standard Import (requires 'solution' package installed)
from solution.src.common import PROCESSED_DATA_DIR
from solution.src.data import load_raw, get_raw_path, PROCESSED_FILES, RAW_FILES
from solution.src.utils import save_refined, print_fancy_progress

# 📦 Import atomic cleaning functions and utils
from nb_01_cleaning import (
    REMAINING_FILES,
    optimize_dtypes,
    get_memory_usage_mb,
    clean_string_columns,
    standardize_dataset,
    resolve_content_duplicates,
    print_cleaning_report,
    print_fk_remapping_report,
    process_ratings_chunk,
    process_generic_dataset
)

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

print(f"Saving to:    {PROCESSED_DATA_DIR}")

# Global storage for ID remapping (to be applied to relational tables)
GLOBAL_ID_MAP = {}


## 1. Characters (`characters.csv`)

**Role**: Establishes the valid universe of Character IDs.

**Processing Steps**:
-   **Deduplication**: Identifies and merges duplicate characters, updating the Global ID Map.
-   **Cleaning**: Drops `about` sections to save space (they're not important for our needs).
-   **Validation**: Captures `valid_character_ids` to filter orphan references in other files.

In [ ]:
# 1. Load
df_chars = load_raw('characters')

# 2. Clean (Conservative)
initial_rows = len(df_chars)

# Remove exact duplicates only
df_chars, std_metrics = standardize_dataset(df_chars)

# Resolve Content Duplicates (Merge IDs)
df_chars, dup_metrics = resolve_content_duplicates(df_chars, primary_key='character_id')

# Store ID map for later use in relational tables
if dup_metrics['id_map']:
    GLOBAL_ID_MAP['character_id'] = dup_metrics['id_map']

# Drop empty rows but NOT just missing IDs
df_chars.dropna(how='all', inplace=True)

df_chars = df_chars.drop(columns=['about'])

rows_dropped_total = initial_rows - len(df_chars)

# 3. Report
report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_chars),
    "Total Dropped": rows_dropped_total
}

print_cleaning_report("Characters", report)

# 4. Optimize & Save
df_chars = optimize_dtypes(df_chars)
save_refined(df_chars, PROCESSED_DATA_DIR / "characters.parquet", format='parquet')

# 5. Capture Valid Character IDs
valid_character_ids = set(df_chars['character_id'].dropna())
print(f"Captured {len(valid_character_ids)} valid Character IDs.")

gc.collect();


## 2. Anime Details (`details.csv`)

**Role**: Establishes the valid universe of Anime IDs.

**Processing Steps**:
-   **Imputation**: Derives missing `year` from `start_date`.
-   **Type Conversion**: Parses dates and converts low-cardinality columns (Genres, Studios) to Categories.
-   **Validation**: Captures `valid_anime_ids`.

In [ ]:
# 1. Load
df_details = load_raw('details')

# 2. Clean (Conservative)
initial_rows = len(df_details)

# Standardize
df_details, std_metrics = standardize_dataset(df_details)

# Resolve Content Duplicates
# Check if resolve_content_duplicates is imported (it should be from previous patch)
# If not, the notebook run might fail, but we assume previous patch worked.
df_details, dup_metrics = resolve_content_duplicates(df_details, primary_key='anime_id')
if dup_metrics['id_map']:
    GLOBAL_ID_MAP['anime_id'] = dup_metrics['id_map']

df_details.dropna(how='all', inplace=True)

# Features
for col in ['start_date', 'end_date']:
    df_details[col] = pd.to_datetime(df_details[col], errors='coerce')

# --- Imputation: meaningful calculations ---
# Calculate Year from start_date if missing
missing_year_before = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
df_details['year'] = df_details['year'].fillna(df_details['start_date'].dt.year)
missing_year_after = df_details['year'].isna().sum() if 'year' in df_details.columns else 0
year_imputed_count = missing_year_before - missing_year_after
# -------------------------------------------

cat_cols = ['type', 'status', 'source', 'rating', 'season']
for col in cat_cols:
    if col in df_details.columns:
        df_details[col] = df_details[col].astype('category')

report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_details),
    "Year Imputed": year_imputed_count
}

print_cleaning_report("Details", report)

# 3. Optimize & Save
df_details = optimize_dtypes(df_details)
save_refined(df_details, PROCESSED_DATA_DIR / "details.parquet", format='parquet')

# 4. Capture Valid Anime IDs
valid_anime_ids = set(df_details['anime_id'].dropna())
print(f"Captured {len(valid_anime_ids)} valid Anime IDs.")

gc.collect();


## 3. Anime Stats (`stats.csv`)

**Role**: Supplementary validation source. Often contains duplicates not present in `details.csv`.

**Processing Steps**:
-   **Merge Duplicates**: Updates the `GLOBAL_ID_MAP` if new duplicates are found.
-   **Referential Integrity**: Ensures all entries map to valid Anime IDs from Step 2.

In [ ]:
# 1. Load
df_stats = load_raw('stats')

# 2. Clean
initial_rows = len(df_stats)

df_stats, std_metrics = standardize_dataset(df_stats)
df_stats, dup_metrics = resolve_content_duplicates(df_stats, primary_key='anime_id')
# Merge maps if any found here (unlikely if unique per anime)
if dup_metrics['id_map']:
    if 'anime_id' not in GLOBAL_ID_MAP:
        GLOBAL_ID_MAP['anime_id'] = {}
    GLOBAL_ID_MAP['anime_id'].update(dup_metrics['id_map'])

df_stats.dropna(how='all', inplace=True)

report = {
    "Initial Rows": initial_rows,
    "Standardization": std_metrics,
    "Content Resolution": dup_metrics,
    "Final Rows": len(df_stats)
}
print_cleaning_report("Stats", report)

# 3. Optimize & Save
df_stats = optimize_dtypes(df_stats)
save_refined(df_stats, PROCESSED_DATA_DIR / "stats.parquet", format='parquet')

gc.collect();


## 4. User Ratings (`ratings.csv`)

**Challenge**: Huge dataset (>120M rows). Cannot process in memory sequentially without being slow.

**Solution: Parallel Streaming Approach**
-   **Parallel Workers**: The file is read in chunks, and each chunk is processed by a separate CPU core.
-   **Streaming Write**: Processed chunks are immediately written to disk (Parquet), keeping RAM usage low.
-   **FK Remapping**: We apply the `GLOBAL_ID_MAP` here to update Anime IDs that were merged in previous steps. 
-   **Orphan Removal**: Votes for non-existent Anime are dropped.

In [ ]:
chunk_size = 5_000_000
csv_path = get_raw_path('ratings')

print(f"Processing {csv_path} in parallel chunks with Conservative Logic...")

id_map_anime = GLOBAL_ID_MAP.get('anime_id', {})

stats = {
    'rows_read': 0,
    'dropped_orphans': 0,
    'fk_remapped': 0,
    'fk_samples': []
}

def _count_lines(fname):
    """Fast line counting (approx 12MB buffer)."""
    with open(fname, 'rb') as f:
        lines = 0
        buf_size = 1024 * 1024 * 12
        read_f = f.read
        buf = read_f(buf_size)
        while buf:
            lines += buf.count(b'\n')
            buf = read_f(buf_size)
    return lines

print("Counting rows for progress estimation...")
total_rows = _count_lines(csv_path) - 1
estimated_total_chunks = (total_rows // chunk_size) + (1 if total_rows % chunk_size > 0 else 0)
print(f"Estimated total rows: {total_rows:,}, Total chunks: {estimated_total_chunks}")

# Generator for Parallel Processing Pipeline
reader = pd.read_csv(csv_path, chunksize=chunk_size)
print("Starting Parallel Processing Pipeline for Ratings huge Dataset...")

# Use joblib to process chunks in parallel
# return_as='generator' allows us to consume results as they finish (streaming),
# which is crucial for writing to Parquet without holding everything in RAM.
processed_chunks = Parallel(n_jobs=-1, return_as='generator')(
    delayed(process_ratings_chunk)(chunk, valid_anime_ids, id_map_anime)
    for chunk in reader
)

print("Optimized parallel processing complete.")

# Consume generator and write to Parquet
# We unzip (chunk, metrics) from the result
def result_generator():
    for i, (processed_chunk, metrics) in enumerate(processed_chunks, 1):
        print_fancy_progress(i, estimated_total_chunks, f'Chunk {i}: ')

        # Aggregate Metrics
        stats['rows_read'] += metrics['rows_read']
        stats['dropped_orphans'] += metrics['dropped_orphans']
        stats['fk_remapped'] += metrics['fk_remapped']
        if len(stats['fk_samples']) < 5:
            stats['fk_samples'].extend(metrics['fk_samples'])

        yield processed_chunk

save_refined(result_generator(), PROCESSED_DATA_DIR / "ratings.parquet", format='parquet')

final_report = {
    "Total Rows Read": stats['rows_read'],
    "Orphans Dropped": stats['dropped_orphans'],
    "FKs Remapped": stats['fk_remapped'],
    "Final Rows Est.": stats['rows_read'] - stats['dropped_orphans']
}

print_cleaning_report("Ratings", final_report)
print_fk_remapping_report(stats['fk_samples'])

gc.collect();


## 5. Remaining Files

**Strategy**: Apply the same rigor to the remaining 9 Relational and Metadata tables.

**Parallel Execution**:
We use `process_generic_dataset` (defined in `utils.py`) to run the cleaning pipeline for all these files **concurrently**. 

**Understanding the Report:**
-   **`Clean` Impact**: Memory usage usually *increases* here (e.g., `-10%` reduction) because we expand efficient storage into Python objects for cleaning.
-   **`Opt` Impact**: Memory usage *decreases* massively (e.g., `+80%` reduction) as we convert to efficient types.
-   **`Total` Impact**: The net result. Usually a significant reduction, ensuring the final Parquet files are highly optimized.

In [ ]:
print(f"Processing {len(REMAINING_FILES)} other files in parallel...")

# Run in parallel
results = Parallel(n_jobs=-1)(
    delayed(process_generic_dataset)(name, PROCESSED_FILES)
    for name in REMAINING_FILES
)

# Print Summary
print(f"\n--- Other Files Cleaning Report ---")
for res in results:
    total_red = (res['mem_src'] - res['mem_dst']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0
    clean_red = (res['mem_src'] - res['mem_cleaned']) / res['mem_src'] * 100 if res['mem_src'] > 0 else 0
    opt_red = (res['mem_cleaned'] - res['mem_dst']) / res['mem_cleaned'] * 100 if res['mem_cleaned'] > 0 else 0

    print(f"[{res['file']}]")
    print(f"  Rows:   {res['rows_src']} -> {res['rows_dst']}")
    print(f"  Memory: {res['mem_src']:.2f} -> {res['mem_cleaned']:.2f} -> {res['mem_dst']:.2f} MB")
    print(f"  Reduc.: {clean_red:.1f}% (Clean) + {opt_red:.1f}% (Opt) = {total_red:.1f}% (Total)")
    if res['actions']:
        print(f"  Actions: {', '.join(res['actions'])}")
    print("-" * 30)
print(f"Total files processed: {len(results)}")

gc.collect();
